In [ ]:
# Python ≥3.9 is required
import sys
assert sys.version_info >= (3, 9)

import seaborn as sns

# Statistical analysis and testing
from scipy import stats
from statsmodels.formula.api import ols
import statsmodels.api as sm

# Ignore useless warnings (see SciPy issue #5998)
import warnings
warnings.filterwarnings(action="ignore", message="^internal gelsd")

In [ ]:
# Python ≥3.8 is required
import sys
assert sys.version_info >= (3, 8)

# Scikit-Learn ≥ 1.0 is required
import sklearn
assert sklearn.__version__ >= "1.0"

# Common imports
import numpy as np
import pandas as pd
import os

# To plot figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Precision options
np.set_printoptions(precision=2)
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
# load in the  dataset
nhanes = pd.read_csv(
    "nhanes.csv"
)

In [ ]:
nhanes.info()

In [ ]:
nhanes.sample(5)

In [ ]:
nhanes.describe()

In [ ]:
# create a training and test set - 20% of the dataset was set away for the test set
from sklearn.model_selection import train_test_split
nhanes = nhanes.dropna(subset=["LBXGH"])
X = nhanes.drop(columns=["LBXGH"])
y = (nhanes["LBXGH"] >= 6.5).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    stratify=y,
    test_size=0.2,
    random_state=42
)

In [ ]:
X_train.shape, X_test.shape

In [ ]:
table_vars = [
    "RIDAGEYR",
    "RIDRETH3",
    "DMDEDUC2",
    "INDFMPIR",
    "BMXBMI",
    "LBXGLU",
    "HUQ010",
    "PAD790Q",
    "PAD810Q"
]

# add LBXGH back
train_table = X_train[table_vars].copy()
train_table["LBXGH"] = nhanes.loc[X_train.index, "LBXGH"]

test_table = X_test[table_vars].copy()
test_table["LBXGH"] = nhanes.loc[X_test.index, "LBXGH"]


In [ ]:
continuous_vars = [
    "RIDAGEYR",
    "INDFMPIR",
    "BMXBMI",
    "LBXGH",
    "LBXGLU",
    "PAD790Q",
    "PAD810Q"
]

train_table[continuous_vars].describe().T

In [ ]:
test_table[continuous_vars].describe().T

In [ ]:
categorical_vars = [
    "RIDRETH3",
    "DMDEDUC2",
    "HUQ010",
    "PAD790Q",
    "PAD810Q"
]

for var in categorical_vars:
    print(f"\n{'='*50}")
    print(var)

    # training set
    train_counts = train_table[var].value_counts(dropna=False)
    train_pct = train_table[var].value_counts(
        normalize=True, dropna=False
    ) * 100

    train_summary = pd.DataFrame({
        "Count": train_counts,
        "Percent": train_pct.round(1)
    })

    print("\nTRAINING:")
    print(train_summary)

    # test set
    test_counts = test_table[var].value_counts(dropna=False)
    test_pct = test_table[var].value_counts(
        normalize=True, dropna=False
    ) * 100

    test_summary = pd.DataFrame({
        "Count": test_counts,
        "Percent": test_pct.round(1)
    })

    print("\nTEST:")
    print(test_summary)


In [ ]:
race_labels = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    6: "Non-Hispanic Asian",
    7: "Other Race"
}

education_labels = {
    1: "Less than 9th grade",
    2: "9-11th grade",
    3: "High school graduate/GED",
    4: "Some college or AA degree",
    5: "College graduate or above",
    7: "Refused",
    9: "Don't know"
}

health_labels = {
    1: "Excellent",
    2: "Very good",
    3: "Good",
    4: "Fair",
    5: "Poor",
    7: "Refused",
    9: "Don't know"
}


train_table["RIDRETH3_label"] = train_table["RIDRETH3"].map(race_labels)
train_table["DMDEDUC2_label"] = train_table["DMDEDUC2"].map(education_labels)
train_table["HUQ010_label"] = train_table["HUQ010"].map(health_labels)

test_table["RIDRETH3_label"] = test_table["RIDRETH3"].map(race_labels)
test_table["DMDEDUC2_label"] = test_table["DMDEDUC2"].map(education_labels)
test_table["HUQ010_label"] = test_table["HUQ010"].map(health_labels)


In [ ]:
categorical_vars = [
    "RIDRETH3_label",
    "DMDEDUC2_label",
    "HUQ010_label"
]

for var in categorical_vars:
    print(f"\n{'='*60}")
    print(var)

    # training
    train_counts = train_table[var].value_counts(dropna=False)
    train_pct = train_table[var].value_counts(
        normalize=True,
        dropna=False
    ) * 100

    train_summary = pd.DataFrame({
        "N": train_counts,
        "%": train_pct.round(1)
    })

    print("\nTRAINING:")
    print(train_summary)

    # test
    test_counts = test_table[var].value_counts(dropna=False)
    test_pct = test_table[var].value_counts(
        normalize=True,
        dropna=False
    ) * 100

    test_summary = pd.DataFrame({
        "N": test_counts,
        "%": test_pct.round(1)
    })

    print("\nTEST:")
    print(test_summary)


In [ ]:
categorical_vars = [
    "RIDRETH3",
    "DMDEDUC2",
    "HUQ010",
    "PAD790Q",
    "PAD810Q"
]

for var in categorical_vars:
    print("\n", var)
    print("TRAINING:")
    print(train_table[var].value_counts(dropna=False))
    print("\nTEST:")
    print(test_table[var].value_counts(dropna=False))


In [ ]:
print("LBXGH >= 6.5%:", (nhanes["LBXGH"] >= 6.5).sum())
print("LBXGH < 6.5%:", (nhanes["LBXGH"] < 6.5).sum())
print("Total:", nhanes["LBXGH"].notna().sum())


In [ ]:
n_high = (nhanes["LBXGH"] >= 6.5).sum()
n_total = nhanes["LBXGH"].notna().sum()

print("LBXGH >= 6.5%:", n_high)
print("Percentage:", round((n_high / n_total) * 100, 2), "%")

In [ ]:
print("Training set:")
print("LBXGH >= 6.5%:", (y_train == 1).sum())
print("LBXGH < 6.5%:", (y_train == 0).sum())
print("Total:", len(y_train))

print("\nTest set:")
print("LBXGH >= 6.5%:", (y_test == 1).sum())
print("LBXGH < 6.5%:", (y_test == 0).sum())
print("Total:", len(y_test))